In [ ]:
# Import Libraries

import pandas as pd
import seaborn as sns
import missingno as msno

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

import scipy.cluster.hierarchy as sch
from sklearn.cluster import AgglomerativeClustering

In [ ]:
# Import the Data

df = pd.read_csv('Ecommerce.zip', encoding='unicode_escape')
df = df.loc[:, ~df.columns.str.startswith('Unnamed:')]
df.columns = df.columns.str.strip().str.upper()
df.head()

In [ ]:
#  Let`s see what the data looks like with the describe() function
df.describe()

In [ ]:
# Checking the missing values

miss_values = df.columns[df.isnull().any()]
print(f"Missing values:\n{df[miss_values].isnull().sum()}")

null_values = df.columns[df.isna().any()]
print(f"Null values:\n{df[null_values].isna().sum()}")

In [ ]:
# Since there are missing values in the dataset, it will be a good idea to see, which columns are affected and will it be a problem for our analysis

null_plot = msno.bar(df, color = "#5F9EA0")

In [ ]:
df_hist_plot = df.hist(figsize = (10,10), color = "#5F9EA0")

In [ ]:
# We filter the table to see where the bulk of the NaN values are and their impact on the output.
# Clearly from the new table and the histogram the results are biased.

purchases = df.loc[df["UNITPRICE"] >= 0].copy()
purchases["Total_Revenue"] = purchases["UNITPRICE"] * purchases["QUANTITY"]
sales = (
    purchases.groupby("CUSTOMERID")
    .agg(
        COUNTRY=("COUNTRY", "first"),
        Sales_Country=("COUNTRY", "count"),
        Total_Revenue=("Total_Revenue", "sum"),
        Total_Items_Sold=("QUANTITY", "sum"),
    )
    .reset_index()
)
sales.head(10)

In [ ]:
sales_hist_plot = sales.hist(figsize = (10,10), color = "#5F9EA0")

In [ ]:
# Total revenue from NaN values makes a huge bias, that needs to be cleared.

top_sales = sales.sort_values("Total_Items_Sold", ascending=False)
top_sales.head(25)

In [ ]:
# After dropping NaN values, the values are more evenly spread now, apart from some outliers.

eliminated_na = top_sales.dropna()
eliminated_na.head()
eliminated_na_hist_plot = eliminated_na.hist(figsize = (10,10), color = "#5F9EA0")

In [ ]:
eliminated_na_hist_plot = eliminated_na.hist(figsize = (10,10), color = "#5F9EA0")

In [ ]:
# Let`s see how the results will be clustering on a graph.

clustering = eliminated_na[["CUSTOMERID", "Sales_Country", "Total_Revenue", "Total_Items_Sold"]]
clustering.head()
plt.scatter(clustering["Total_Items_Sold"], clustering["Total_Revenue"])

In [ ]:
# The bulk of the values are stuck around the (0.0) point of the graph. To help with the better finds the cluster centers, we will set limitations to the top values of revenue and sales.

clustering = clustering.query("Total_Revenue < 30000 and Total_Items_Sold < 100000")
plt.scatter(clustering["Total_Items_Sold"], clustering["Total_Revenue"])

In [ ]:
# We use Elbow method to find the best number of K - clusters.

cluster_columns = ['Total_Items_Sold', 'Total_Revenue']
scaler = MinMaxScaler()
points = scaler.fit_transform(clustering[cluster_columns])

Sum_of_squared_distances = []
K = range(1, 15)
for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km = km.fit(points)
    Sum_of_squared_distances.append(km.inertia_)

In [ ]:
plt.plot(K, Sum_of_squared_distances, 'bx-')
plt.xlabel('k')
plt.ylabel('Sum_of_squared_distances')
plt.title('Elbow Method For Optimal k')
plt.show()

print(km.cluster_centers_)

In [ ]:
# We create an array with the columns with most weight to predict the clusters.

km = KMeans(n_clusters=6, random_state=42, n_init=10)
y_km = km.fit_predict(points)
y_km

In [ ]:
plt.scatter(points[y_km ==0,0], points[y_km == 0,1], s=100, c='red')
plt.scatter(points[y_km ==1,0], points[y_km == 1,1], s=100, c='black')
plt.scatter(points[y_km ==2,0], points[y_km == 2,1], s=100, c='blue')
plt.scatter(points[y_km ==3,0], points[y_km == 3,1], s=100, c='green')
plt.scatter(points[y_km ==4,0], points[y_km == 4,1], s=100, c='yellow')
plt.scatter(points[y_km ==5,0], points[y_km == 5,1], s=100, c='pink')

plt.xlabel('Total Items Sold')
plt.ylabel('Total Revenue')
plt.title('Clustering the Customers')

In [ ]:
# create dendrogram
dendrogram = sch.dendrogram(sch.linkage(points, method='ward'))
# create clusters
hc = AgglomerativeClustering(n_clusters=6, metric='euclidean', linkage='ward')
# save clusters for chart
y_hc = hc.fit_predict(points)

In [ ]:
plt.scatter(points[y_hc ==0,0], points[y_hc == 0,1], s=100, c='red')
plt.scatter(points[y_hc==1,0], points[y_hc == 1,1], s=100, c='black')
plt.scatter(points[y_hc ==2,0], points[y_hc == 2,1], s=100, c='blue')
plt.scatter(points[y_hc ==3,0], points[y_hc == 3,1], s=100, c='green')
plt.scatter(points[y_hc ==4,0], points[y_hc == 4,1], s=100, c='yellow')
plt.scatter(points[y_hc ==5,0], points[y_hc == 5,1], s=100, c='pink')

plt.xlabel('Total Items Sold')
plt.ylabel('Total Revenue')
plt.title('Clustering the Customers')

In [ ]:
clustering["cluster"] = km.labels_
clustering.head(25)